In [1]:
import optuna
import torch
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from torchmetrics import F1Score
import warnings
warnings.filterwarnings('ignore')

import dotenv
import os
dotenv.load_dotenv(dotenv_path="./../src/GradientGang/Pipeline/Optimizer/.env")
storage = (os.getenv("DATABASE_URL"))

from GradientGang.Pipeline.DataLoader.DataLoader import DataModule
from GradientGang.Pipeline.Architectures.LightningAutoencoder import LightningAutoencoder

## Optuna Hyperparameter Tuning for Autoencoder

This notebook implements hyperparameter optimization using Optuna for an Autoencoder architecture with CNN encoder/decoder.

## 1. Setup Data Loaders

Keep data loading parameters fixed to focus on model architecture hyperparameters.

In [2]:
# Fixed data loading parameters
data_params = {
    'data_dir': "../dataset/PirateProcessed/",
    'train_file_name': "pirate_pain_train.csv",
    'train_file_name_labels': "pirate_pain_train_labels.csv",
    'test_file_name': "pirate_pain_test.csv",
    'batch_size': 32,
    'num_workers': 0,
    'val_split': 0.2,
    'shuffle': True,
}

# Initialize data module
dataLoader = DataModule(params=data_params)
dataLoader.setup(stage='fit', includeTestInTrain=True)

trainLoader = dataLoader.train_dataloader()
valLoader = dataLoader.val_dataloader()

print("Data loaders initialized successfully!")
print(f"Training batches: {len(trainLoader)}")
print(f"Validation batches: {len(valLoader)}")

Data loaders initialized successfully!
Training batches: 58
Validation batches: 5


## 2. Define Architecture Parameter Generation

Create functions to generate architecture parameters for Autoencoder model based on Optuna trial suggestions.

In [3]:
def create_architecture_params(trial):
    """
    Create architecture parameters based on Optuna trial suggestions for Autoencoder.
    
    Parameters to tune:
    - Learning rate
    - Regularization weight
    - Reconstruction loss weight
    - CNN encoder parameters (channels, kernel sizes)
    - Latent dimension (embedding size)
    - Activation functions
    - Feed-forward hidden layer sizes
    - Dropout rates
    """
    
    # Suggest hyperparameters
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    reg_weight = trial.suggest_float('regularization_weight', 1e-6, 1e-2, log=True)
    reconstruction_loss_weight = trial.suggest_float('reconstruction_loss_weight', 0.01, 1.0, log=True)
    
    # Encoder CNN parameters
    conv1_channels = trial.suggest_categorical('conv1_channels', [32, 64, 128])
    conv2_channels = trial.suggest_categorical('conv2_channels', [64, 128, 256])
    conv3_channels = trial.suggest_categorical('conv3_channels', [32, 64, 128])
    
    # Fixed kernel sizes and padding for stable reconstruction
    conv1_kernel = 10
    conv2_kernel = 3
    conv3_kernel = 3
    
    pool_kernel = 2
    pool_stride = 2
    
    # Latent dimension
    latent_dim = trial.suggest_categorical('latent_dim', [64, 128, 256, 512])
    
    # Activation functions
    encoder_activation = trial.suggest_categorical('encoder_activation', ['ReLU', 'GELU', 'LeakyReLU'])
    decoder_activation = trial.suggest_categorical('decoder_activation', ['ReLU', 'GELU', 'LeakyReLU'])
    ff_activation = trial.suggest_categorical('ff_activation', ['ReLU', 'GELU', 'LeakyReLU'])
    
    # Feed-forward parameters
    ff_num_layers = trial.suggest_int('ff_num_layers', 1, 3)
    ff_hidden_size = trial.suggest_categorical('ff_hidden_size', [32, 64, 128, 256])
    ff_dropout = trial.suggest_float('ff_dropout', 0.0, 0.5)
    
    # Patience for early stopping
    patience = trial.suggest_int('patience', 20, 50)
    
    # Calculate sequence lengths after each layer
    # Original: 160
    # After Conv1d(kernel=10, stride=1, padding=1): 160 - 10 + 2*1 + 1 = 153
    # After MaxPool1d(kernel=2, stride=2): (153 - 2) / 2 + 1 = 76
    seq_after_pool = 76
    
    # Fixed adaptive pool size for consistent reconstruction
    adaptive_pool_size = 76
    
    # Calculate flattened size after encoder
    flattened_size = conv3_channels * adaptive_pool_size
    
    # Build encoder layer configuration
    encoder_layers = [
        {
            "name": "Conv1d",
            "params": {
                "in_channels": 34,
                "out_channels": conv1_channels,
                "kernel_size": conv1_kernel,
                "stride": 1,
                "padding": 1,
                "bias": True,
            },
        },
        {
            "name": "MaxPool1d",
            "params": {
                "kernel_size": pool_kernel,
                "stride": pool_stride,
                "padding": 0,
                "dilation": 1,
                "return_indices": False,
                "ceil_mode": False
            }
        },
        {
            "name": "Conv1d",
            "params": {
                "in_channels": conv1_channels,
                "out_channels": conv2_channels,
                "kernel_size": conv2_kernel,
                "stride": 1,
                "padding": 1,
                "bias": True,
            },
        },
        {
            "name": "Conv1d",
            "params": {
                "in_channels": conv2_channels,
                "out_channels": conv3_channels,
                "kernel_size": conv3_kernel,
                "stride": 1,
                "padding": 1,
                "bias": True,
            },
        },
        {
            "name": "AdaptiveAvgPool1d",
            "params": { "output_size": adaptive_pool_size }
        },
        {
            "name": "Flatten",
            "params": {}
        },
        {
            "name": "Linear",
            "params": {
                "in_features": flattened_size, 
                "out_features": latent_dim,
                "bias": True,
            }
        }
    ]
    
    # Build decoder layer configuration (mirrors encoder)
    # We need to go from latent_dim back to (batch, 34, 160)
    # Strategy: Use AdaptiveAvgPool1d at the end to ensure exact size
    decoder_layers = [
        {
            "name": "Linear",
            "params": {
                "in_features": latent_dim,
                "out_features": flattened_size,
                "bias": True,
            }
        },
        {
            "name": "Unflatten",
            "params": {
                "dim": 1,
                "unflattened_size": [conv3_channels, adaptive_pool_size]
            }
        },
        # Reverse conv3: maintain same size with padding=1
        {
            "name": "ConvTranspose1d",
            "params": {
                "in_channels": conv3_channels,
                "out_channels": conv2_channels,
                "kernel_size": conv3_kernel,
                "stride": 1,
                "padding": 1,
                "output_padding": 0,
                "groups": 1,
                "dilation": 1,
                "padding_mode": "zeros",
                "bias": True,
            }
        },
        # Reverse conv2: maintain same size with padding=1
        {
            "name": "ConvTranspose1d",
            "params": {
                "in_channels": conv2_channels,
                "out_channels": conv1_channels,
                "kernel_size": conv2_kernel,
                "stride": 1,
                "padding": 1,
                "output_padding": 0,
                "groups": 1,
                "dilation": 1,
                "padding_mode": "zeros",
                "bias": True,
            }
        },
        # Reverse maxpool: upsample from 76 to 152 (76*2)
        {
            "name": "ConvTranspose1d",
            "params": {
                "in_channels": conv1_channels,
                "out_channels": conv1_channels,
                "kernel_size": pool_kernel,
                "stride": pool_stride,
                "padding": 0,
                "output_padding": 0,
                "groups": 1,
                "dilation": 1,
                "padding_mode": "zeros",
                "bias": True,
            }
        },
        # Reverse conv1: expand channels
        {
            "name": "ConvTranspose1d",
            "params": {
                "in_channels": conv1_channels,
                "out_channels": 34,
                "kernel_size": conv1_kernel,
                "stride": 1,
                "padding": 1,
                "output_padding": 0,
                "groups": 1,
                "dilation": 1,
                "padding_mode": "zeros",
                "bias": True,
            }
        },
        # Use AdaptiveAvgPool1d to ensure exact output size of 160
        {
            "name": "AdaptiveAvgPool1d",
            "params": { "output_size": 160 }
        }
    ]
    
    # Build feed-forward layer configuration
    ff_layers = []
    current_size = latent_dim + 1  # latent_dim + global_feature_size
    
    for i in range(ff_num_layers):
        # Add Linear layer
        ff_layers.append({
            "name": "Linear",
            "params": {
                "in_features": current_size,
                "out_features": ff_hidden_size,
                "bias": True,
            }
        })
        
        # Add Dropout layer (not after the last one)
        if ff_dropout > 0 and i < ff_num_layers - 1:
            ff_layers.append({
                "name": "Dropout",
                "params": {
                    "p": ff_dropout
                }
            })
        
        current_size = ff_hidden_size
    
    # Build architecture parameters
    architecture_params = {
        "LearningRate": learning_rate,
        "Patience": patience,
        "RegularizationWeight": reg_weight,
        "ReconstructionLossWeight": reconstruction_loss_weight,
        "ClassWeightsPath": "../dataset/PirateProcessed/class_weights.yaml",
        "EncoderParams": {
            "activation_function": encoder_activation,
            "layer_type": encoder_layers
        },
        "DecoderParams": {
            "activation_function": decoder_activation,
            "layer_type": decoder_layers
        },
        "FeedForwardParams": {
            "activation_function": ff_activation,
            "layer_type": ff_layers
        },
        "GlobalFFEncoderParams": {
            "activation_function": "LeakyReLU",
            "layer_type": [
                {
                    "name": "Linear",
                    "params": {
                        "in_features": 1,
                        "out_features": 1,
                        "bias": True,
                    }
                },
            ]
        },
        "GlobalFFDecoderParams": {
            "activation_function": "LeakyReLU",
            "layer_type": [
                {
                    "name": "Linear",
                    "params": {
                        "in_features": 1,
                        "out_features": 1,
                        "bias": True,
                    }
                },
            ]
        },
        "OutputDim": 3
    }
    
    return architecture_params

## 3. Define Objective Function

The objective function trains the Autoencoder model and returns the validation F1 score.

In [4]:
def objective(trial):
    """
    Objective function for Optuna optimization.
    Returns validation F1 score to maximize.
    """
    
    # Generate architecture parameters from trial
    architecture_params = create_architecture_params(trial)
    
    # Create model
    model = LightningAutoencoder(architecture_params)
    
    # Training parameters
    max_epochs = 300
    
    # Add early stopping callback
    early_stopping_callback = EarlyStopping(
        monitor='val_F1',
        patience=architecture_params["Patience"],
        mode='max',  # We want to maximize F1 score
        verbose=False
    )

    # Save best model checkpoint
    checkpoint_callback = ModelCheckpoint(
        monitor='val_F1',
        mode='max',
        save_top_k=5,
        filename='autoencoder-best-{epoch:02d}-{val_F1:.3f}',
        verbose=True
    )

    # TensorBoard logger per salvare pesi, grafo e metriche
    logger = TensorBoardLogger(
        save_dir='lightning_logs',
        name='autoencoder_optuna',
        version=f'trial_{trial.number}',  # Usa il numero del trial come versione
        log_graph=True,  # Salva il grafo del modello
        default_hp_metric=False
    )

    trainer = Trainer(
        max_epochs=max_epochs,
        enable_progress_bar=False,
        enable_model_summary=False,
        log_every_n_steps=20,
        callbacks=[early_stopping_callback, checkpoint_callback],
        logger=logger  # Aggiungi il logger
    )
    
    # Train the model
    try:
        trainer.fit(model, trainLoader, valLoader)
    except Exception as e:
        print(f"Trial {trial.number} failed with error: {e}")
        return 0.0
    
    # Evaluate on validation set
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in valLoader:
            x, labels = batch
            # Forward returns (predictions, (decoded_timeseries, decoded_global))
            preds, _ = model(x)
            all_preds.extend(preds.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate F1 score
    f1_metric = F1Score(task="multiclass", num_classes=3, average='weighted')
    f1_score = f1_metric(torch.tensor(all_preds), torch.tensor(all_labels))
    
    # Report intermediate value for pruning
    trial.report(f1_score.item(), step=trainer.current_epoch)
    
    # Handle pruning
    if trial.should_prune():
        raise optuna.TrialPruned()
    
    return f1_score.item()

## 4. Create and Run Optuna Study

Configure the study with pruning and run optimization for Autoencoder.

In [5]:
# Empty the folder of lightning_logs/ before starting new optimization
import shutil
from pathlib import Path

logs_dir = Path('lightning_logs')
if logs_dir.exists():
    shutil.rmtree(logs_dir)
    print(f"Removed existing lightning_logs directory")
logs_dir.mkdir(exist_ok=True)
print("Created fresh lightning_logs directory")

Created fresh lightning_logs directory


In [ ]:
# Create study
study = optuna.create_study(
    direction='maximize',  # We want to maximize F1 score
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
    study_name='autoencoder_cnn_hyperparameter_tuning',
    storage=storage,
    load_if_exists=True
)

# Run optimization
n_trials = 100  # Number of trials to run
print(f"Starting optimization...")

study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

print("\nOptimization complete!")

[I 2025-11-12 15:49:21,725] Using an existing study with name 'autoencoder_cnn_hyperparameter_tuning' instead of creating a new one.


Starting optimization...


  0%|          | 0/100 [00:00<?, ?it/s]

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Epoch 0, global step 58: 'val_F1' reached 0.68182 (best 0.68182), saving model to 'lightning_logs\\autoencoder_optuna\\trial_0\\checkpoints\\autoencoder-best-epoch=00-val_F1=0.682.ckpt' as top 5
Epoch 0, global step 58: 'val_F1' reached 0.68182 (best 0.68182), saving model to 'lightning_logs\\autoencoder_optuna\\trial_0\\checkpoints\\autoencoder-best-epoch=00-val_F1=0.682.ckpt' as top 5
Epoch 1, global step 116: 'val_F1' reached 0.74242 (best 0.74242), saving model to 'lightning_logs\\autoencoder_optuna\\trial_0\\checkpoints\\autoencoder-best-epoch=01-val_F1=0.742.ckpt' as top 5
Epoch 1, global step 116: 'val_F1' reached 0.74242 (best 0.74242), saving model to 'lightning_logs\\autoencoder_optuna\\trial_0\\checkpoints\\autoencoder-best-epoch=01-val_F1=0.742.ckpt' as top 5
Epoch 2, global step 174: 'val

[I 2025-11-12 15:50:47,930] Trial 0 finished with value: 0.8400283455848694 and parameters: {'learning_rate': 0.0009282971430751124, 'regularization_weight': 2.2321986079524602e-05, 'reconstruction_loss_weight': 0.22287832111679354, 'conv1_channels': 32, 'conv2_channels': 64, 'conv3_channels': 64, 'latent_dim': 256, 'encoder_activation': 'LeakyReLU', 'decoder_activation': 'ReLU', 'ff_activation': 'GELU', 'ff_num_layers': 2, 'ff_hidden_size': 64, 'ff_dropout': 0.02224579340618199, 'patience': 21}. Best is trial 0 with value: 0.840028345584869.


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Epoch 0, global step 58: 'val_F1' reached 0.13636 (best 0.13636), saving model to 'lightning_logs\\autoencoder_optuna\\trial_1\\checkpoints\\autoencoder-best-epoch=00-val_F1=0.136.ckpt' as top 5
Epoch 0, global step 58: 'val_F1' reached 0.13636 (best 0.13636), saving model to 'lightning_logs\\autoencoder_optuna\\trial_1\\checkpoints\\autoencoder-best-epoch=00-val_F1=0.136.ckpt' as top 5
Epoch 1, global step 116: 'val_F1' reached 0.13636 (best 0.13636), saving model to 'lightning_logs\\autoencoder_optuna\\trial_1\\checkpoints\\autoencoder-best-epoch=01-val_F1=0.136.ckpt' as top 5
Epoch 1, global step 116: 'val_F1' reached 0.13636 (best 0.13636), saving model to 'lightning_logs\\autoencoder_optuna\\trial_1\\checkpoints\\autoencoder-best-epoch=01-val_F1=0.136.ckpt' as top 5
Epoch 2, global step 174: 'val

[I 2025-11-12 15:53:51,722] Trial 1 finished with value: 0.7484256625175476 and parameters: {'learning_rate': 1.8273381848208614e-05, 'regularization_weight': 0.009760622895793693, 'reconstruction_loss_weight': 0.27810482719346463, 'conv1_channels': 32, 'conv2_channels': 64, 'conv3_channels': 32, 'latent_dim': 512, 'encoder_activation': 'GELU', 'decoder_activation': 'GELU', 'ff_activation': 'GELU', 'ff_num_layers': 3, 'ff_hidden_size': 256, 'ff_dropout': 0.23427852950181005, 'patience': 38}. Best is trial 0 with value: 0.840028345584869.


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Epoch 0, global step 58: 'val_F1' reached 0.68182 (best 0.68182), saving model to 'lightning_logs\\autoencoder_optuna\\trial_2\\checkpoints\\autoencoder-best-epoch=00-val_F1=0.682.ckpt' as top 5
Epoch 0, global step 58: 'val_F1' reached 0.68182 (best 0.68182), saving model to 'lightning_logs\\autoencoder_optuna\\trial_2\\checkpoints\\autoencoder-best-epoch=00-val_F1=0.682.ckpt' as top 5
Epoch 1, global step 116: 'val_F1' reached 0.68182 (best 0.68182), saving model to 'lightning_logs\\autoencoder_optuna\\trial_2\\checkpoints\\autoencoder-best-epoch=01-val_F1=0.682.ckpt' as top 5
Epoch 1, global step 116: 'val_F1' reached 0.68182 (best 0.68182), saving model to 'lightning_logs\\autoencoder_optuna\\trial_2\\checkpoints\\autoencoder-best-epoch=01-val_F1=0.682.ckpt' as top 5
Epoch 2, global step 174: 'val

[I 2025-11-12 15:56:44,537] Trial 2 finished with value: 0.848141074180603 and parameters: {'learning_rate': 6.105957616752677e-05, 'regularization_weight': 0.007187233319349995, 'reconstruction_loss_weight': 0.031150011013784284, 'conv1_channels': 64, 'conv2_channels': 256, 'conv3_channels': 64, 'latent_dim': 64, 'encoder_activation': 'ReLU', 'decoder_activation': 'GELU', 'ff_activation': 'ReLU', 'ff_num_layers': 3, 'ff_hidden_size': 128, 'ff_dropout': 0.014541421588303405, 'patience': 22}. Best is trial 2 with value: 0.848141074180603.


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Epoch 0, global step 58: 'val_F1' reached 0.68182 (best 0.68182), saving model to 'lightning_logs\\autoencoder_optuna\\trial_3\\checkpoints\\autoencoder-best-epoch=00-val_F1=0.682.ckpt' as top 5
Epoch 0, global step 58: 'val_F1' reached 0.68182 (best 0.68182), saving model to 'lightning_logs\\autoencoder_optuna\\trial_3\\checkpoints\\autoencoder-best-epoch=00-val_F1=0.682.ckpt' as top 5
Epoch 1, global step 116: 'val_F1' reached 0.68939 (best 0.68939), saving model to 'lightning_logs\\autoencoder_optuna\\trial_3\\checkpoints\\autoencoder-best-epoch=01-val_F1=0.689.ckpt' as top 5
Epoch 1, global step 116: 'val_F1' reached 0.68939 (best 0.68939), saving model to 'lightning_logs\\autoencoder_optuna\\trial_3\\checkpoints\\autoencoder-best-epoch=01-val_F1=0.689.ckpt' as top 5
Epoch 2, global step 174: 'val

## 5. Analyze Results

Display the best parameters and trial statistics.

In [7]:
print("BEST TRIAL RESULTS")
print(f"\nBest F1 Score: {study.best_trial.value:.4f}")
print(study.best_trial._number)
print("\nBest Hyperparameters:")
study.best_trial.params


BEST TRIAL RESULTS


ValueError: Record does not exist.

In [ ]:
# Load the best model from checkpoint
checkpoint_path = "lightning_logs/version_261/checkpoints/autoencoder-best-epoch=43-val_F1=0.955.ckpt"

# Recreate the architecture params from the best trial
best_architecture_params = create_architecture_params(study.best_trial)

# Load model with the correct parameter name
best_model = LightningAutoencoder.load_from_checkpoint(checkpoint_path, params=best_architecture_params)
print(f"Loaded best model from: {checkpoint_path}")

Loaded best model from: lightning_logs/version_261/checkpoints/autoencoder-best-epoch=43-val_F1=0.955.ckpt


In [ ]:
# Reload SubmissionGenerator to get the latest changes
import importlib
import GradientGang.Pipeline.SubmissionGenerator.SubmissionGenerator
importlib.reload(GradientGang.Pipeline.SubmissionGenerator.SubmissionGenerator)
from GradientGang.Pipeline.SubmissionGenerator.SubmissionGenerator import SubmissionGenerator

dataLoader.setup(stage='test')
testLoader = dataLoader.test_dataloader()

submission_generator = SubmissionGenerator(
    model=best_model,
    dataloader=testLoader,
    label_mapping={0: 'no_pain', 1: 'low_pain', 2: 'high_pain'},
)

from datetime import datetime
path = "../Submissions/submission_AE_" + datetime.now().strftime("%H-%M") + ".csv"
submission_generator.generate_submission(
    output_path = path
)

,sample_index,label
0,000,no_pain
1,001,no_pain
2,002,no_pain
3,003,no_pain
4,004,no_pain
...,...,...
1319,1319,high_pain
1320,1320,high_pain
1321,1321,no_pain
1322,1322,no_pain
